# 03. 자동 Stage 1 연구 경로 (선택)

`01`의 레거시 기준선을 선택적 자동 연구 루프로 탐색하는 과정을 기록합니다.

```text
diagnosis -> bounded proposal -> AIDM -> evidence verification -> human review
```

이 경로는 탐색 단계만 제한적으로 자동화합니다. 진단, 하나의 제한된 JSON 제안, AIDM 실험, 증적 검증까지 자동으로 수행한 뒤 **`ready_for_human_review`에서 멈춥니다.** AIDD 호출, 코드 생성, 고객 시스템 변경, 병합, 배포는 하지 않습니다.

- 저장소 루트에서 실행하세요.
- 합성 fixture 설정을 사용하며 출력은 `.agents/runs/research-loop-fixture/`에 남습니다.
- 이 루프는 게이트 임계값을 바꾸지 않고, 고객 행·타깃·비밀값을 기록하지 않습니다.

In [ ]:
import shutil
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd()
RUNNER = REPO_ROOT / ".agents" / "scripts" / "run-research-loop.sh"
CONFIG = REPO_ROOT / ".agents" / "fixtures" / "research-loop.json"
RUN_DIR = REPO_ROOT / ".agents" / "runs" / "research-loop-fixture"

# 데모를 반복 실행할 수 있도록 로컬 run 디렉터리를 정리합니다(.agents/runs 는 로컬 전용).
shutil.rmtree(RUN_DIR, ignore_errors=True)

# research-orchestrator: 진단 -> 제안 -> AIDM -> 검증 상태 머신을 안전하게 실행합니다.
completed = subprocess.run(
    [str(RUNNER), "--config", str(CONFIG)],
    cwd=REPO_ROOT,
    text=True,
    capture_output=True,
    check=True,
)
print("연구 루프 실행 완료")
print(sorted(path.name for path in RUN_DIR.iterdir()))

In [ ]:
import json

diagnosis = json.loads((RUN_DIR / "diagnosis.json").read_text(encoding="utf-8"))
summary = json.loads((RUN_DIR / "research-summary.json").read_text(encoding="utf-8"))

# research-diagnostic: 집계 전용 데이터 품질·누수 진단 (원시 행은 남기지 않음).
print("[진단] 누수 검사:")
for check, passed in diagnosis["leakage_checks"].items():
    print(f"  - {check}: {passed}")
print(f"[진단] 추천 프로필: {diagnosis['recommended_profiles']}")

# research-verification: AIDM 증적을 게이트 변경 없이 재검증한 결과.
print(f"\n[요약] 상태: {summary['status']}")
print(f"[요약] 반복 횟수: {summary['iterations']}")
print(f"[요약] 사용한 프로필: {summary['used_profiles']}")
print(f"[요약] 검증 결과: {summary['verifier']['outcome']}")

## 사람 검토 경계

상태가 `ready_for_human_review`에서 멈췄습니다. `research-summary.json`은 연구 진단·제안·검증 결과일 뿐 release 또는 deploy 승인 증거가 **아닙니다.**

사람은 이 요약과 전체 증적(SHA-256으로 연결된 `diagnosis.json`, iteration별 `promotion_manifest.json`·`verification.json`)을 검토한 뒤, `02` 노트북의 수동 AIDD 및 `release-gate` 절차를 **별도로** 실행해야 합니다.

| 단계 | 스킬 | 반드시 멈추는 경계 |
| --- | --- | --- |
| 상태 머신 조정 | `research-orchestrator` | AIDD·코드 생성·고객 변경·병합·배포 전에 종료 |
| 집계 진단 | `research-diagnostic` | 후보를 발명하거나 AIDM·AIDD를 실행하지 않음 |
| 제한된 제안 생성 | `research-proposal` | 코드·임의 탐색·게이트 변경을 만들지 않음 |
| 증적 재검증 | `research-verification` | AIDM 재실행·release 승인·AIDD 호출을 하지 않음 |

즉, 자동 경로는 사람이 검토할 **후보와 증적을 준비**할 뿐, 개선을 확정하거나 배포하지 않습니다.